In [1]:
!pip install -q gradio

In [3]:
import torch
import torchaudio

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MODEL_PATH = "/content/drive/MyDrive/audio-separation/checkpoints/best_conv_tasnet.pth"

model = torchaudio.models.conv_tasnet_base(
    num_sources=2
)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=DEVICE
    )
)

model = model.to(DEVICE)
model.eval()

print("Model loaded on:", DEVICE)

Model loaded on: cpu


In [4]:
import torchaudio
import torch

SAMPLE_RATE = 16000


def separate_audio(audio_path):

    waveform, sample_rate = torchaudio.load(
        audio_path
    )

    # Stereo → mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(
            dim=0,
            keepdim=True
        )

    # Resample
    if sample_rate != SAMPLE_RATE:

        resampler = torchaudio.transforms.Resample(
            sample_rate,
            SAMPLE_RATE
        )

        waveform = resampler(waveform)

    # Move to GPU
    waveform = waveform.to(DEVICE)

    # [1, samples] → [1, 1, samples]
    mixture = waveform.unsqueeze(0)

    with torch.no_grad():

        estimates = model(mixture)

    # [1, 2, samples] → [2, samples]
    estimates = estimates.squeeze(0)

    speech = estimates[0]
    noise = estimates[1]

    speech_path = "/content/speech.wav"
    noise_path = "/content/noise.wav"

    torchaudio.save(
        speech_path,
        speech.unsqueeze(0).cpu(),
        SAMPLE_RATE
    )

    torchaudio.save(
        noise_path,
        noise.unsqueeze(0).cpu(),
        SAMPLE_RATE
    )

    return speech_path, noise_path

In [5]:
import gradio as gr


def process_audio(audio):

    speech, noise = separate_audio(audio)

    return speech, noise


demo = gr.Interface(
    fn=process_audio,

    inputs=gr.Audio(
        type="filepath",
        label="Upload Noisy Audio"
    ),

    outputs=[
        gr.Audio(
            label="Clean Speech"
        ),
        gr.Audio(
            label="Background Noise"
        )
    ],

    title="Audio Separation Engine",

    description=(
        "Upload a noisy audio recording and "
        "separate speech from background noise "
        "using Conv-TasNet."
    )
)


demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7eb354d67b54b2ae70.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
